# Pydantic AI + Neo4j Aura Agent

The [main notebook](./pydantic_ai.ipynb) ran an MCP server next to the agent and pointed it at a
database. An **Aura Agent** moves that whole layer into Aura: you configure it once in the console -
which instance it reads, what it knows about the data model, how it should answer - and Neo4j hosts
it behind an MCP endpoint.

What changes for you:

| | Self-hosted MCP server | Aura Agent |
| --- | --- | --- |
| Where it runs | Your process | Neo4j Aura |
| What it exposes | Raw Cypher tools | A configured agent over your data |
| Who writes the Cypher | Your model, each time | The hosted agent, from its own configuration |
| Auth | HTTP Basic, database credentials | OAuth 2.0 client credentials |
| Schema knowledge | Rediscovered per conversation | Configured once, in the console |

For Pydantic AI it is still just a toolset. Everything else in this notebook - dependency
injection, structured output, combining toolsets - works exactly as it does locally.

> **The agent used below is one I configured in my own Aura project.** The ids are placeholders:
> create an agent in [the Aura console](https://console.neo4j.io), point it at an instance of yours,
> and drop its project and agent ids in. Nothing else in the notebook changes.

## 1. What you need

Two things from the [Aura console](https://console.neo4j.io):

**1. An agent.** Create one against an instance, give it instructions and whatever data-model hints
it needs. Its URL contains the two ids you need:
`console.neo4j.io/projects/<project_id>/agents/<agent_id>`.

**2. Client credentials - the right kind.** Account settings → **Client credentials** →
**Aura Agent & MCP**. These are *not* the Aura API keys used for provisioning instances. The two
look identical and fail in a way that does not say so: API keys are issued for `api.neo4j.io`, the
MCP endpoint trusts a different issuer, and using the wrong pair returns
`Jwt issuer is not configured`.

In [1]:
%pip install -q --upgrade "pydantic-ai>=2.0" requests


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

AURA_MCP_TOKEN_URL = "https://mcp.neo4j.io/oauth/token"
AURA_MCP_AUDIENCE = "https://agent-mcp.neo4j.io"

CLIENT_ID = os.environ.get("AURA_MCP_CLIENT_ID") or getpass("Aura MCP client ID: ")
CLIENT_SECRET = os.environ.get("AURA_MCP_CLIENT_SECRET") or getpass("Aura MCP client secret: ")
AGENT_MCP_URL = os.environ.get("AURA_AGENT_MCP_URL") or input("Agent MCP endpoint URL: ").strip()

MODEL = "openai:gpt-5.4-mini"
# Show the endpoint without leaking the IDs into notebook output.
print("Endpoint:", AGENT_MCP_URL.split("?")[0], "(project and agent IDs hidden)")

Endpoint: https://mcp.neo4j.io/agent (project and agent IDs hidden)


## 2. The access token

Machine-to-machine OAuth: exchange the client id and secret for a bearer token, send that token with
every MCP request. No browser, no redirect - which is what makes this usable from a backend or a
scheduled job.

**The token endpoint is rate limited to 15 requests per hour per client id.** That is generous for a
service that caches and stingy if you fetch one per call, so the cache below is not an optimisation
- rerun a cell a few times without it and you will spend the rest of the hour locked out.

In [5]:
import time

import requests

_token_cache = {"value": None, "expires_at": 0.0}


def get_token(force: bool = False) -> str:
    """Return a cached bearer token, refreshing it only when it is close to expiry."""
    if not force and _token_cache["value"] and time.time() < _token_cache["expires_at"] - 60:
        return _token_cache["value"]

    response = requests.post(
        AURA_MCP_TOKEN_URL,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "audience": AURA_MCP_AUDIENCE,
        },
        timeout=30,
    )
    response.raise_for_status()

    payload = response.json()
    _token_cache["value"] = payload["access_token"]
    _token_cache["expires_at"] = time.time() + payload.get("expires_in", 3600)
    return _token_cache["value"]


token = get_token()
print(f"Token acquired, valid for {int(_token_cache['expires_at'] - time.time())}s")

Token acquired, valid for 86399s


## 3. Verify the endpoint before wiring it to an agent

If discovery fails, most frameworks hand the agent an
empty tool list and carry on, and the agent's polite "I don't have access to that" reads like a
prompt problem.

Whatever the hosted agent chooses to expose shows up here - the tool list comes from the agent's
configuration, not from a fixed contract.

In [7]:
from pydantic_ai.mcp import MCPToolset

aura_agent_tools = MCPToolset(
    AGENT_MCP_URL,
    headers={"Authorization": f"Bearer {token}"},
    id="aura-agent",
)

async with aura_agent_tools:
    advertised = await aura_agent_tools.list_tools()

print("Aura Agent advertises:")
for tool in advertised:
    summary = (tool.description or "").strip().splitlines()[:1]
    print(f"  • {tool.name} - {summary[0] if summary else 'no description'}")

assert advertised, "No tools discovered - check the project id, agent id and credentials."
print("\nAura Agent ready")

Aura Agent advertises:
  • Investment_Support_Agent - Assists with exploring companies, their partnerships, key personnel, and relevant articles for investment purposes.

Aura Agent ready


## 4. Using it

Nothing here is Aura-specific. It is a toolset, so it goes in the list like any other.

Notice what you do *not* write: no schema instructions, no Cypher guidance, no "call get-schema
first". The hosted agent already knows its own data model, so your instructions can be about the
task rather than the database.

In [9]:
from pydantic_ai import Agent

analyst = Agent(
    MODEL,
    toolsets=[aura_agent_tools],
    instructions=(
        "You are a research assistant with access to a Neo4j knowledge graph through a hosted "
        "agent. Ask it for what you need, then answer the user plainly. If it returns nothing "
        "useful, say so rather than guessing."
    ),
)

async with analyst:
    result = await analyst.run("What kinds of entities and relationships does this graph contain?")

print(result.output)

The graph appears to contain these main entity types:

- **Organization** — companies, corporations, institutions
- **Person** — executives, board members, stakeholders
- **IndustryCategory** — sectors or market categories
- **Country** and **City** — geographic locations
- **Article** — news or report documents
- **Chunk** — smaller text segments from articles

And these main relationship types:

- **`HAS_PARENT` / `HAS_SUBSIDIARY` / `HAS_CHILD`** — corporate hierarchy
- **`HAS_CEO` / `HAS_BOARD_MEMBER`** — leadership and governance
- **`HAS_SUPPLIER` / `HAS_COMPETITOR` / `HAS_INVESTOR`** — business relationships
- **`HAS_CATEGORY`** — links organizations to industry categories
- **`IN_COUNTRY` / `IN_CITY`** — geographic placement
- **`MENTIONS`** — articles referencing organizations
- **`HAS_CHUNK`** — articles connected to their text chunks

So overall, it’s a corporate knowledge graph covering **companies, people, locations, industry classification, and media coverage**.


In [10]:
async with analyst:
    result = await analyst.run(
        "Who are the competitors of OpenAI in the AI industry?"
    )

print(result.output)
print("\nTools called:", [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if part.part_kind == "tool-call"
])

Major competitors of OpenAI in the AI industry include:

- Google / DeepMind — strong competitor in foundation models with Gemini, plus AI across search and productivity
- Meta Platforms — competing with open-weight Llama models and developer ecosystem
- Cohere — focused on enterprise LLMs and business use cases
- Amazon / AWS — competes through Bedrock, Titan models, and cloud AI infrastructure
- Alibaba — major AI competitor, especially in Asia, with Qwen and cloud AI services
- Midjourney — competes in generative image AI specifically

If you want, I can also break these down by:
1. direct ChatGPT competitors,
2. enterprise AI competitors, or
3. open-source model competitors.

Tools called: ['Investment_Support_Agent']


## 5. Hosted knowledge, local tools

The useful shape is rarely "hosted agent alone". It is a hosted agent for what the graph knows,
plus local tools for what only your process knows — a portfolio, a user's permissions, today's
prices, an internal API.

Pydantic AI composes them in one list, and the model sees a single flat set of tools. Your
dependencies stay local: the hosted agent never sees `deps`.

In [12]:
from dataclasses import dataclass, field

from pydantic_ai import RunContext
from pydantic_ai.toolsets import FunctionToolset


@dataclass
class PortfolioDeps:
    """Local state the hosted agent has no knowledge of."""
    holdings: dict[str, float] = field(default_factory=dict)
    cash: float = 0.0


portfolio_tools = FunctionToolset(
    id="portfolio",
    instructions="Use the portfolio tools for anything about what the user currently holds.",
)


@portfolio_tools.tool
async def get_holdings(ctx: RunContext[PortfolioDeps]) -> dict:
    """The user's current positions and available cash."""
    return {"holdings": ctx.deps.holdings, "cash": ctx.deps.cash}


@portfolio_tools.tool
async def position_size(ctx: RunContext[PortfolioDeps], company: str) -> str:
    """How much of the portfolio is in one company.

    Args:
        company: Company name to look up.
    """
    value = ctx.deps.holdings.get(company)
    if value is None:
        return f"No position in {company}."
    total = sum(ctx.deps.holdings.values()) + ctx.deps.cash
    return f"{company}: {value:,.0f} of {total:,.0f} ({value / total:.1%} of the portfolio)."


combined = Agent(
    MODEL,
    deps_type=PortfolioDeps,
    toolsets=[aura_agent_tools, portfolio_tools],
    instructions=(
        "You advise on a small portfolio. The hosted graph agent knows about companies and "
        "their relationships; the portfolio tools know what the user actually holds. Use both "
        "and be explicit about which facts came from where."
    ),
)

deps = PortfolioDeps(
    holdings={"Microsoft": 42_000, "Nvidia": 31_000, "Neo4j": 8_000},
    cash=19_000,
)

async with combined:
    result = await combined.run(
        "How concentrated is my position in Nvidia, and what does the graph say about who it "
        "competes with?",
        deps=deps,
    )

print(result.output)

Your Nvidia position is **31,000 out of 100,000**, so it’s **31.0% of your portfolio**.  
- **Source:** portfolio tool (`position_size`)

That’s a **fairly concentrated** single-name position: about **1/3 of the portfolio** in one stock.

From the company graph, Nvidia’s direct competitors include:  
- **AMD**
- **Intel**
- **Qualcomm**
- **Texas Instruments**
- **Xilinx** (now part of AMD)
- **ATI Technologies** (historical)

The graph’s context says Nvidia competes mainly in:  
- **High-performance computing / AI / data center chips**
- **Gaming GPUs**
- **Specialized semiconductor markets** like automotive and edge AI

- **Source:** graph agent (`Investment_Support_Agent`)

If you want, I can also help think through what a 31% Nvidia position means for portfolio risk.


## 6. A typed answer on top

The hosted agent returns text. `output_type` puts a schema in front of it, so what reaches your
code is a validated object rather than a paragraph to parse.

In [13]:
from typing import Literal

from pydantic import BaseModel, Field


class Finding(BaseModel):
    claim: str = Field(description="One fact, stated plainly.")
    source: Literal["graph", "portfolio"] = Field(description="Where the fact came from.")


class Assessment(BaseModel):
    summary: str = Field(description="Two or three sentences.")
    findings: list[Finding]
    concentration_risk: Literal["low", "medium", "high"]
    next_step: str = Field(description="One concrete thing the user could do.")


assessor = Agent(
    MODEL,
    deps_type=PortfolioDeps,
    output_type=Assessment,
    toolsets=[aura_agent_tools, portfolio_tools],
    instructions=(
        "Assess the portfolio using the graph for company relationships and the portfolio "
        "tools for positions. Tag every finding with where it came from."
    ),
)

async with assessor:
    result = await assessor.run("Assess my portfolio for concentration risk.", deps=deps)

assessment = result.output
print(f"{assessment.summary}\n")
print(f"Concentration risk: {assessment.concentration_risk}\n")
for finding in assessment.findings:
    print(f"  [{finding.source}] {finding.claim}")
print(f"\nNext step: {assessment.next_step}")

Your portfolio is highly concentrated in two large tech names: Microsoft is 42% and Nvidia is 31%, with Neo4j at 8% and cash at 19%. The graph suggests Microsoft and Nvidia share exposure to the broader AI/cloud ecosystem, so your risk is more about one theme than about independent positions.

Concentration risk: high

  [portfolio] Microsoft is 42.0% of the portfolio.
  [portfolio] Nvidia is 31.0% of the portfolio.
  [graph] Microsoft and Nvidia are both exposed to the same AI/cloud ecosystem, creating cluster risk.
  [graph] Neo4j is a software layer that depends on cloud-native integration, including Microsoft Azure.

Next step: Consider trimming one of the two largest positions or adding exposure to a less correlated sector to reduce single-theme concentration.


## Summary

| | |
| --- | --- |
| Endpoint | `https://mcp.neo4j.io/agent?project_id=…&agent_id=…` |
| Token endpoint | `https://mcp.neo4j.io/oauth/token` |
| Audience | `https://agent-mcp.neo4j.io` |
| Grant | `client_credentials` (no browser, works from a backend) |
| Credentials | Console → Account settings → Client credentials → **Aura Agent & MCP** |
| In Pydantic AI | `MCPToolset(url, headers={"Authorization": f"Bearer {token}"})` |

### Resources

- [Neo4j Aura console](https://console.neo4j.io)
- [Pydantic AI documentation](https://ai.pydantic.dev/)
- [Main notebook](./pydantic_ai.ipynb) - self-hosted MCP, custom tools, approval, memory
- [GraphRAG notebook](./pydantic_ai_graphrag.ipynb) - retrieval over the same kind of graph